This chapter is entirely about answering one question:

Our ML model predicts P1–P4. How do we convert that into a Probability of Default required by IFRS 9?

### ***1. Probability of Default (PD) Mapping***

In [4]:
# ==========================================================
# Chapter 1 : Probability of Default (PD) Mapping
# ==========================================================

# Mapping Internal Risk Grades to Probability of Default (PD)

PD_MAPPING = {
    "P1": {
        "description": "Very Low Risk",
        "pd": 0.01
    },
    "P2": {
        "description": "Low Risk",
        "pd": 0.03
    },
    "P3": {
        "description": "Medium Risk",
        "pd": 0.08
    },
    "P4": {
        "description": "High Risk",
        "pd": 0.20
    }
}

In [5]:
# Reusable Function To Calculate PD based on The Risk grades
def calculate_pd(risk_grade: str) -> float:
    """
    Convert an internal risk grade (P1-P4)
    into its corresponding Probability of Default (PD).

    Parameters
    ----------
    risk_grade : str
        Predicted internal credit risk grade.

    Returns
    -------
    float
        Probability of Default.
    """

    risk_grade = risk_grade.upper()

    if risk_grade not in PD_MAPPING:
        raise ValueError(
            f"Invalid Risk Grade '{risk_grade}'. "
            "Expected one of: P1, P2, P3 or P4."
        )

    return PD_MAPPING[risk_grade]

In [8]:
# Testing the Function
print("P1 ->", calculate_pd("P1")['pd'])
print("P2 ->", calculate_pd("P2")['pd'])
print("P3 ->", calculate_pd("P3")['pd'])
print("P4 ->", calculate_pd("P4")['pd'])

P1 -> 0.01
P2 -> 0.03
P3 -> 0.08
P4 -> 0.2


We want to display additional information in the Streamlit dashboard, such as:

- Risk Grade : P3
- Description : Medium Risk
- Probability of Default : 8%

With the nested dictionary, all related information stays together, making it easier to extend the system in the future without changing the overall structure.

===========================================================================================================================

# 2: Loss Given Default (LGD)

## What is LGD?

LGD (Loss Given Default) is the percentage of the loan that the bank expects to lose if the borrower defaults.

The bank does not always lose the entire loan amount because it can recover some money by selling the collateral (security) provided by the borrower.

---

## Example

- Loan Amount = ₹8,00,000
- Collateral Value = ₹10,00,000

Collateral Coverage:

Coverage = Collateral Value / Loan Amount

= 10,00,000 / 8,00,000

= **125%**

Since the collateral value is greater than the loan amount, the bank has a lower chance of losing money.

Therefore, we assign a **lower LGD (20%)**.

---

## Why are we using calibrated values?

Our dataset does not contain:

- Recovery history
- Legal costs
- Auction prices
- Historical LGD values

So, instead of predicting LGD using Machine Learning, we use **business calibration rules**.

These rules simulate how a bank estimates potential losses.

---

## LGD Calibration Table

| Collateral Coverage | LGD |
|--------------------:|----:|
| ≥ 100% | 20% |
| 80% – 99% | 35% |
| 50% – 79% | 50% |
| 20% – 49% | 70% |
| < 20% | 90% |

---

## Workflow

Credit Officer enters:

- Loan Amount
- Bank-Assessed Collateral Value

↓

System calculates:

Collateral Coverage

↓

LGD Engine checks the calibration table

↓

Returns the appropriate LGD value

---

## Why did we build this module?

This module estimates how much money the bank may lose if the borrower defaults.

The calculated LGD will be used in the next chapter to calculate the **Expected Credit Loss (ECL)** using the IFRS 9 formula.

In [9]:
# ==========================================================
# Chapter 2 : Loss Given Default (LGD)
# ==========================================================

# LGD Calibration Rules based on Collateral Coverage

LGD_RULES = [
    {"min_coverage": 1.00, "lgd": 0.20},
    {"min_coverage": 0.80, "lgd": 0.35},
    {"min_coverage": 0.50, "lgd": 0.50},
    {"min_coverage": 0.20, "lgd": 0.70},
    {"min_coverage": 0.00, "lgd": 0.90}
]

In [10]:
# LGD Function
def calculate_lgd(
    loan_amount: float,
    collateral_value: float
):
    """
    Calculate Loss Given Default (LGD)
    using collateral coverage.

    Parameters
    ----------
    loan_amount : float
        Requested loan amount.

    collateral_value : float
        Bank-assessed collateral value.

    Returns
    -------
    tuple
        (LGD, Coverage Ratio)
    """

    if loan_amount <= 0:
        raise ValueError("Loan amount must be greater than zero.")

    coverage = collateral_value / loan_amount

    for rule in LGD_RULES:
        if coverage >= rule["min_coverage"]:
            return rule["lgd"], coverage

    return 0.90, coverage

In [11]:
# Test the Function
examples = [
    (800000, 1000000),
    (800000, 700000),
    (800000, 500000),
    (800000, 250000),
    (800000, 50000)
]

for loan, collateral in examples:

    lgd, coverage = calculate_lgd(
        loan,
        collateral
    )

    print(f"Loan Amount      : ₹{loan:,}")
    print(f"Collateral Value : ₹{collateral:,}")
    print(f"Coverage Ratio   : {coverage:.2%}")
    print(f"LGD              : {lgd:.0%}")

    print("-" * 45)

Loan Amount      : ₹800,000
Collateral Value : ₹1,000,000
Coverage Ratio   : 125.00%
LGD              : 20%
---------------------------------------------
Loan Amount      : ₹800,000
Collateral Value : ₹700,000
Coverage Ratio   : 87.50%
LGD              : 35%
---------------------------------------------
Loan Amount      : ₹800,000
Collateral Value : ₹500,000
Coverage Ratio   : 62.50%
LGD              : 50%
---------------------------------------------
Loan Amount      : ₹800,000
Collateral Value : ₹250,000
Coverage Ratio   : 31.25%
LGD              : 70%
---------------------------------------------
Loan Amount      : ₹800,000
Collateral Value : ₹50,000
Coverage Ratio   : 6.25%
LGD              : 90%
---------------------------------------------


=========================================================================================================================
# 3: Exposure at Default (EAD)

## What is EAD?!

**Exposure at Default (EAD)** is the amount of money the bank is exposed to if the borrower defaults on the loan.

In simple words:

> **EAD represents the total amount the bank stands to lose at the time of default before considering any recoveries.**

---

# Example

Suppose a borrower requests a loan of:

**₹8,00,000**

If the borrower defaults immediately after the loan is sanctioned, the bank's exposure is:

**EAD = ₹8,00,000**

If, in a real banking scenario, the borrower had already repaid part of the loan before defaulting, the EAD would be the **remaining outstanding loan amount**.

---

# Why are we not calculating Outstanding Balance?

Our project is designed as a **Loan Approval Decision Support System**.

The credit officer uses this application **before approving and disbursing the loan**.

At this stage:

- The borrower has only requested a loan.
- No loan has been sanctioned yet.
- No EMI has been paid.
- No repayment history exists.
- Therefore, there is no outstanding balance to calculate.

---

# System Design Decision

Since our application evaluates the borrower **before loan approval**, we make the following business assumption:

> **Exposure at Default (EAD) = Requested Loan Amount**

This assumption is appropriate because the requested loan amount represents the bank's maximum initial exposure if the loan is approved.

---

# Workflow

Credit Officer enters:

- Requested Loan Amount

↓

EAD Engine

↓

Returns:

**EAD = Requested Loan Amount**

---

# Why did we choose this approach?

The original dataset does not contain:

- Outstanding Loan Balance
- Repayment History
- EMI Information
- Credit Conversion Factors
- Undrawn Loan Commitments

Therefore, it is not possible to estimate a dynamic EAD.

Using the **requested loan amount as the initial exposure** is a realistic and transparent assumption for a loan origination system.

---

# Real Banking vs Our Project

### Real Banking System

EAD is calculated using:

- Outstanding Principal
- Accrued Interest
- Undrawn Credit Limits
- Credit Conversion Factors (CCF)

### Our Project

Since the loan is evaluated **before approval**, we assume:

**EAD = Requested Loan Amount**

This accurately represents the bank's initial financial exposure at the loan origination stage.

---

# Summary

- EAD measures the bank's exposure if the borrower defaults.
- Our application is designed for **pre-loan approval**.
- Since no repayments have occurred yet, there is no outstanding balance.
- Therefore, **Requested Loan Amount = Exposure at Default (EAD)**.
- This EAD value will be used in the next chapter to calculate **Expected Credit Loss (ECL)** using the IFRS 9 formula.

In [12]:
# ==========================================================
# Chapter 3 : Exposure at Default (EAD)
# ==========================================================

def calculate_ead(loan_amount: float) -> float:
    """
    Calculate Exposure at Default (EAD).

    For this project, EAD is assumed to be equal
    to the requested loan amount.

    Parameters
    ----------
    loan_amount : float
        Requested loan amount.

    Returns
    -------
    float
        Exposure at Default.
    """

    if loan_amount <= 0:
        raise ValueError("Loan amount must be greater than zero.")

    return loan_amount

In [13]:
loan_amount = 800000

ead = calculate_ead(loan_amount)

print(f"Loan Amount : ₹{loan_amount:,}")
print(f"EAD         : ₹{ead:,}")

Loan Amount : ₹800,000
EAD         : ₹800,000


=========================================================================================================================
# 4: Expected Credit Loss (ECL)

## What is Expected Credit Loss (ECL)?

Expected Credit Loss (ECL) is the amount of money the bank expects to lose if a borrower defaults on a loan.

Unlike traditional accounting, IFRS 9 requires banks to estimate **future credit losses in advance**, even before a borrower actually defaults.

---

# IFRS 9 Formula

The Expected Credit Loss is calculated using the following formula:

**ECL = PD × LGD × EAD**

Where:

- **PD (Probability of Default):** The likelihood that the borrower will default.
- **LGD (Loss Given Default):** The percentage of the loan expected to be lost after recovering collateral.
- **EAD (Exposure at Default):** The bank's financial exposure at the time of default.

---

# Example

Suppose a borrower has:

- **PD = 3%**
- **LGD = 35%**
- **Requested Loan Amount = ₹8,00,000**

Since our system evaluates the loan **before approval**, we assume:

**EAD = Requested Loan Amount = ₹8,00,000**

Now calculate:

ECL = 0.03 × 0.35 × 800000

= **₹8,400**

This means that although the bank is lending ₹8,00,000, based on the customer's risk profile and collateral, the bank expects an average loss of **₹8,400**.

---

# System Design

Our IFRS 9 Risk Engine works in the following sequence:

Customer Profile

↓

Machine Learning Model

↓

Predicted Risk Grade (P1–P4)

↓

PD Engine

↓

LGD Engine

↓

EAD Engine

↓

ECL Calculator

↓

Expected Credit Loss (₹)

---

# Why is ECL important?

Expected Credit Loss is one of the most important outputs required under IFRS 9.

Banks use ECL to:

- Estimate future credit losses.
- Maintain adequate financial provisions.
- Support lending decisions.
- Manage portfolio risk.
- Comply with IFRS 9 regulatory requirements.

---

# Why did we build the ECL Engine?

Our Machine Learning model predicts only the customer's **risk category (P1–P4)**.

The IFRS 9 Risk Engine converts that prediction into meaningful financial information by combining:

- PD (Risk of default)
- LGD (Expected percentage loss)
- EAD (Financial exposure)

The ECL Engine then calculates the expected monetary loss for the bank.

---

# Summary

- ECL estimates how much money the bank expects to lose from a loan.
- It combines three components:
  - Probability of Default (PD)
  - Loss Given Default (LGD)
  - Exposure at Default (EAD)
- Our project calculates ECL using the IFRS 9 formula:

**ECL = PD × LGD × EAD**

- The calculated ECL will be displayed in the Streamlit dashboard to help the credit officer make informed lending decisions.

In [14]:
# ==========================================================
# Chapter 4 : Expected Credit Loss (ECL)
# ==========================================================

def calculate_ecl(
    pd: float,
    lgd: float,
    ead: float
) -> float:
    """
    Calculate Expected Credit Loss (ECL).

    Parameters
    ----------
    pd : float
        Probability of Default.

    lgd : float
        Loss Given Default.

    ead : float
        Exposure at Default.

    Returns
    -------
    float
        Expected Credit Loss.
    """

    if not (0 <= pd <= 1):
        raise ValueError("PD must be between 0 and 1.")

    if not (0 <= lgd <= 1):
        raise ValueError("LGD must be between 0 and 1.")

    if ead < 0:
        raise ValueError("EAD cannot be negative.")

    return pd * lgd * ead

In [16]:
# Testing the Function
pd = 0.03
lgd = 0.35
ead = 800000

ecl = calculate_ecl(pd, lgd, ead)

print(f"PD  : {pd:.0%}")
print(f"LGD : {lgd:.0%}")
print(f"EAD : ₹{ead:,.0f}")
print("-" * 35)
print(f"Expected Credit Loss : ₹{ecl:,.2f}")

PD  : 3%
LGD : 35%
EAD : ₹800,000
-----------------------------------
Expected Credit Loss : ₹8,400.00


===========================================================================================================================
# Chapter 5: IFRS 9 Stage Classification

## What is Stage Classification?

Under IFRS 9, every loan is assigned to a **stage** based on its credit quality.

The assigned stage helps the bank determine:

- The level of credit risk.
- The amount of expected credit loss to recognize.
- The monitoring required for the borrower.

---

# The Three IFRS 9 Stages

| Stage | Meaning | Risk Level |
|--------|---------|------------|
| Stage 1 | Performing Loan | Low Risk |
| Stage 2 | Significant Increase in Credit Risk | Medium Risk |
| Stage 3 | Credit-Impaired / High Risk Borrower | High Risk |

---

# How does our project classify stages?

Unlike real banking systems, our dataset does not contain operational loan servicing information such as:

- Days Past Due (DPD)
- Monthly repayment history
- Loan restructuring history
- Bankruptcy indicators

However, our Machine Learning model has already analyzed the borrower's historical credit behaviour using features such as:

- Credit utilization
- Credit enquiries
- Trade line information
- Delinquencies
- Credit history
- Existing liabilities

Based on these historical features, the model predicts an **internal risk grade (P1–P4)**.

Therefore, in our project, the predicted risk grade acts as a summary of the borrower's overall credit quality.

Instead of introducing additional assumptions, we directly use the predicted risk grade to determine the IFRS 9 stage.

---

# Stage Mapping Used in Our Project

| Predicted Risk Grade | IFRS 9 Stage |
|----------------------|--------------|
| P1 | Stage 1 |
| P2 | Stage 1 |
| P3 | Stage 2 |
| P4 | Stage 3 |

---

# System Design

Customer Historical Data

↓

Machine Learning Model

↓

Predicted Risk Grade (P1–P4)

↓

IFRS 9 Stage Classification

↓

Stage 1 / Stage 2 / Stage 3

---

# Why did we choose this approach?

Our Machine Learning model already evaluates the borrower's historical financial behaviour and summarizes it into an internal risk rating.

Therefore, instead of using additional operational variables that are not available in our dataset, we use the predicted risk grade as the primary indicator of credit quality.

This keeps the system:

- Simple
- Explainable
- Consistent with our dataset
- Suitable for an enterprise loan approval decision support system

---

# Summary

- IFRS 9 requires every loan to be assigned to a stage.
- Our Machine Learning model predicts the borrower's internal risk grade (P1–P4).
- The predicted risk grade summarizes the borrower's historical credit behaviour.
- Therefore, we directly map the predicted risk grade to the corresponding IFRS 9 stage.

Final Mapping:

- **P1 → Stage 1**
- **P2 → Stage 1**
- **P3 → Stage 2**
- **P4 → Stage 3**

In [17]:
# ==========================================================
# Chapter 5 : IFRS 9 Stage Classification
# ==========================================================

STAGE_MAPPING = {
    "P1": "Stage 1",
    "P2": "Stage 1",
    "P3": "Stage 2",
    "P4": "Stage 3"
}

In [18]:
def classify_stage(risk_grade: str) -> str:
    """
    Map an internal risk grade (P1-P4)
    to the corresponding IFRS 9 stage.
    """

    risk_grade = risk_grade.upper()

    if risk_grade not in STAGE_MAPPING:
        raise ValueError(
            f"Invalid Risk Grade '{risk_grade}'. "
            "Expected one of: P1, P2, P3 or P4."
        )

    return STAGE_MAPPING[risk_grade]

In [19]:
for grade in ["P1", "P2", "P3", "P4"]:
    stage = classify_stage(grade)
    print(f"{grade} → {stage}")

P1 → Stage 1
P2 → Stage 1
P3 → Stage 2
P4 → Stage 3


==========================================================================================================================
# 6: Lending Decision Support Engine

## Objective

The purpose of this module is to convert the technical outputs of the IFRS 9 Risk Engine into meaningful business decisions that can assist a credit officer during loan approval.

Instead of displaying only numerical values such as PD, LGD, EAD, ECL and IFRS 9 Stage, the system generates a lending recommendation based on the borrower's overall risk profile.

---

# Inputs to the Decision Engine

The Decision Engine receives the following outputs from previous modules:

- Predicted Risk Grade (P1–P4)
- Probability of Default (PD)
- Loss Given Default (LGD)
- Exposure at Default (EAD)
- Expected Credit Loss (ECL)
- IFRS 9 Stage Classification

---

# Why do we need another module?

Although the IFRS 9 Risk Engine calculates several important risk metrics, a credit officer ultimately needs a clear recommendation rather than a collection of numbers.

For example, instead of showing only:

- PD = 8%
- LGD = 50%
- ECL = ₹32,000

the system should provide a business recommendation such as:

- Approve
- Approve with Monitoring
- Approve with Additional Collateral
- Manual Review Required
- Decline Application

This makes the system practical for real-world decision making.

---

# Decision Categories

Our project uses five lending decisions.

| Decision | Meaning |
|-----------|---------|
| Approve | Low-risk borrower with acceptable expected loss |
| Approve with Monitoring | Loan can be approved but requires periodic monitoring |
| Approve with Additional Collateral | Additional security is recommended before approval |
| Manual Review Required | Credit committee or senior officer should review the application |
| Decline Application | Credit risk is considered too high |

---

# Decision Strategy

There are two possible ways to generate lending recommendations.

### Option 1

Use only the predicted risk grade.

This approach ignores the financial impact of the loan.

---

### Option 2 (Selected)

Use:

- IFRS 9 Stage Classification
- Expected Loss Ratio

Expected Loss Ratio is calculated as:

Expected Loss Ratio = ECL / EAD

This provides a normalized measure of expected loss regardless of the requested loan amount.

---

# Why are we using Expected Loss Ratio?

Consider two borrowers.

### Borrower A

Loan Amount = ₹2,00,000

Expected Credit Loss = ₹20,000

Expected Loss Ratio = 10%

---

### Borrower B

Loan Amount = ₹50,00,000

Expected Credit Loss = ₹20,000

Expected Loss Ratio = 0.4%

Although both borrowers have the same Expected Credit Loss in rupees, the financial impact relative to the loan amount is very different.

Using the Expected Loss Ratio allows fair comparison between loans of different sizes.

---

# System Design

Customer Profile

↓

Machine Learning Model

↓

Predicted Risk Grade

↓

PD Engine

↓

LGD Engine

↓

EAD Engine

↓

Expected Credit Loss

↓

Expected Loss Ratio

↓

Decision Engine

↓

Final Lending Recommendation

---

# Outputs

The Decision Engine will generate:

- Lending Decision
- Business Reason
- Monitoring Requirement
- Additional Recommendations

These outputs will be displayed in the Streamlit dashboard to assist the credit officer during loan approval.

---

# Summary

The Lending Decision Support Engine is the final business layer of the IFRS 9 Risk Engine.

It transforms technical risk metrics into actionable lending recommendations.

Instead of requiring the credit officer to interpret multiple financial metrics manually, the system provides a clear and explainable recommendation based on the borrower's predicted risk and expected financial impact on the bank.

In [23]:
DECISION_RULES = [
    {
        "stage": "Stage 1",
        "max_loss_ratio": 0.02,
        "decision": "Approve",
        "monitoring": "Standard Annual Review",
        "reason": "Low expected credit loss and strong borrower credit profile."
    },
    {
        "stage": "Stage 1",
        "max_loss_ratio": 0.05,
        "decision": "Approve with Monitoring",
        "monitoring": "Semi-Annual Review",
        "reason": "Moderate expected credit loss. Regular monitoring is recommended."
    },
    {
        "stage": "Stage 2",
        "max_loss_ratio": 0.05,
        "decision": "Approve with Additional Collateral",
        "monitoring": "Quarterly Review",
        "reason": "Additional collateral is recommended to reduce the bank's potential loss."
    },
    {
        "stage": "Stage 2",
        "max_loss_ratio": 1.00,
        "decision": "Manual Review Required",
        "monitoring": "Senior Credit Committee",
        "reason": "High expected credit loss requires manual credit assessment."
    },
    {
        "stage": "Stage 3",
        "max_loss_ratio": 1.00,
        "decision": "Decline Application",
        "monitoring": "Not Applicable",
        "reason": "Credit risk exceeds the bank's acceptable lending policy."
    }
]

In [28]:
# building the decision Engine
def generate_decision(
    stage: str,
    ecl: float,
    ead: float
):
    """
    Generate lending recommendation based on
    IFRS 9 Stage and Expected Loss Ratio.

    Parameters
    ----------
    stage : str
        IFRS 9 Stage.

    ecl : float
        Expected Credit Loss.

    ead : float
        Exposure at Default.

    Returns
    -------
    dict
        Lending recommendation.
    """

    if ead <= 0:
        raise ValueError("EAD must be greater than zero.")

    loss_ratio = ecl / ead

    for rule in DECISION_RULES:

        if (
            stage == rule["stage"]
            and loss_ratio <= rule["max_loss_ratio"]
        ):

            return {
                "Loss Ratio": loss_ratio,
                "Decision": rule["decision"],
                "Monitoring": rule["monitoring"],
                "Reason":rule["reason"]
            }

    return {
        "Loss Ratio": loss_ratio,
        "Decision": "Manual Review Required",
        "Monitoring": "Senior Credit Committee",
        "Reason":rule["reason"]
    }

In [29]:
# Testing the Engine
examples = [

    ("Stage 1", 8000, 800000),

    ("Stage 1", 25000, 800000),

    ("Stage 2", 30000, 800000),

    ("Stage 2", 100000, 800000),

    ("Stage 3", 50000, 800000)

]

for stage, ecl, ead in examples:

    result = generate_decision(stage, ecl, ead)

    print("=" * 50)

    print(f"Stage      : {stage}")
    print(f"ECL        : ₹{ecl:,.0f}")
    print(f"EAD        : ₹{ead:,.0f}")
    print(f"Loss Ratio : {result['Loss Ratio']:.2%}")
    print(f"Decision   : {result['Decision']}")
    print(f"Monitoring : {result['Monitoring']}")
    print(f"Business-Reasoning : {result['Reason']}")

Stage      : Stage 1
ECL        : ₹8,000
EAD        : ₹800,000
Loss Ratio : 1.00%
Decision   : Approve
Monitoring : Standard Annual Review
Business-Reasoning : Low expected credit loss and strong borrower credit profile.
Stage      : Stage 1
ECL        : ₹25,000
EAD        : ₹800,000
Loss Ratio : 3.12%
Decision   : Approve with Monitoring
Monitoring : Semi-Annual Review
Business-Reasoning : Moderate expected credit loss. Regular monitoring is recommended.
Stage      : Stage 2
ECL        : ₹30,000
EAD        : ₹800,000
Loss Ratio : 3.75%
Decision   : Approve with Additional Collateral
Monitoring : Quarterly Review
Business-Reasoning : Additional collateral is recommended to reduce the bank's potential loss.
Stage      : Stage 2
ECL        : ₹100,000
EAD        : ₹800,000
Loss Ratio : 12.50%
Decision   : Manual Review Required
Monitoring : Senior Credit Committee
Business-Reasoning : High expected credit loss requires manual credit assessment.
Stage      : Stage 3
ECL        : ₹50,000
EA

### 7. Risk Report Generation

In [30]:
# ==========================================================
# Chapter 7 : Risk Report Generator
# ==========================================================

RISK_LEVEL_MAPPING = {
    "P1": "Very Low Risk",
    "P2": "Low Risk",
    "P3": "Moderate Risk",
    "P4": "High Risk"
}

In [41]:
def generate_risk_report(
    applicant_id,
    risk_grade,
    loan_amount,
    collateral_value
):
    risk_level = RISK_LEVEL_MAPPING[risk_grade]
    pd = calculate_pd(risk_grade)['pd']
    lgd, coverage = calculate_lgd(
        loan_amount,
        collateral_value
    )
    ead = calculate_ead(loan_amount)
    ecl = calculate_ecl(
        pd,
        lgd,
        ead
    )
    stage = classify_stage(risk_grade)
    decision = generate_decision(
        stage,
        ecl,
        ead
    )
    return {

        "Applicant ID": applicant_id,

        "Risk Grade": risk_grade,

        "Risk Level": risk_level,

        "PD": pd,

        "LGD": lgd,

        "Loan Amount":loan_amount,

        "Collateral Coverage": coverage,

        "EAD": ead,

        "ECL": ecl,

        "IFRS Stage": stage,

        "Decision": decision["Decision"],

        "Monitoring": decision["Monitoring"],

        "Reason": decision["Reason"],

        "Loss Ratio": decision["Loss Ratio"]

    }

In [43]:
# ==========================================================
# Test : Risk Report Generator
# ==========================================================

report = generate_risk_report(
    applicant_id=100245,
    risk_grade="P3",
    loan_amount=800000,
    collateral_value=600000
)

print("=" * 60)

for key, value in report.items():

    if isinstance(value, (int,float)):

        if key in ["PD", "LGD", "Collateral Coverage", "Loss Ratio"]:
            print(f"{key:<22}: {value:.2%}")

        elif key in ["Loan Amount","EAD", "ECL"]:
            print(f"{key:<22}: ₹{value:,.2f}")

        else:
            print(f"{key:<22}: {value}")

    else:
        print(f"{key:<22}: {value}")

Applicant ID          : 100245
Risk Grade            : P3
Risk Level            : Moderate Risk
PD                    : 8.00%
LGD                   : 50.00%
Loan Amount           : ₹800,000.00
Collateral Coverage   : 75.00%
EAD                   : ₹800,000.00
ECL                   : ₹32,000.00
IFRS Stage            : Stage 2
Decision              : Approve with Additional Collateral
Monitoring            : Quarterly Review
Reason                : Additional collateral is recommended to reduce the bank's potential loss.
Loss Ratio            : 4.00%
